[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navjotts/ML-experiments/blob/master/9%20-%20High%20Dimensionality%20and%20Manual%20Clustering/High_Dimensionality_and_Manual_Clustering.ipynb)

# Purpose
The purpose of this experiment is to showcase

1.   how High Dimensionality makes it harder to pull out any meaning from data ([Curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality)) 
2.   a manual approach to Unsupervised Clustering


We dabble with the Titanic data, and try to find some clusters **manually** - in an attempt to find any meaningful information about the main prediction which concerns the Titanic tragedy - `died v/s survived`.


---



### Data

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

titanic = sns.load_dataset('titanic')
titanic = titanic.drop(['alive','adult_male','who','class','embark_town'], axis=1)
titanic['embarked'] = titanic['embarked'].fillna(method='ffill')
titanic = titanic.drop(['deck'], axis=1)
titanic['age'] = titanic['age'].fillna(method='ffill')
for label in ['embarked', 'sex', 'alone']:
    titanic[label] = LabelEncoder().fit_transform(titanic[label])
    
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,alone
0,0,3,1,22.0,1,0,7.2500,2,0
1,1,1,0,38.0,1,0,71.2833,0,0
2,1,3,0,26.0,0,0,7.9250,2,1
3,1,1,0,35.0,1,0,53.1000,2,0
4,0,3,1,35.0,0,0,8.0500,2,1


# Part 1

### As a First step, let's

1. Calculate the centroid of the survivors
2. Calculate the centroid of the casualties
3. Calculate the average distance between each survivor
4. Calculate the average distance between each casualty
5. Calculate the distance between the two centroids


In [0]:
# 1st separate out survivors and casualities datasets
survivors = titanic[titanic['survived']==1]
casualities = titanic[titanic['survived']==0]

survivors = survivors.drop(['survived'], axis=1)
casualities = casualities.drop(['survived'], axis=1)

features = titanic.drop(['survived'], axis=1)

In [0]:
# centroid of a given data set in a given axes
def centroid(data, axes):
  centroid = []  
  for axis in axes:
    points = data[axis]
    centroid.append(points.mean())  
  return centroid

In [4]:
# centroid of the survivors
survivors_centroid = centroid(survivors, features.columns.values)
print(survivors_centroid)

[1.9502923976608186, 0.31871345029239767, 28.373070175438595, 0.47368421052631576, 0.4649122807017544, 48.39540760233918, 1.3596491228070176, 0.4766081871345029]


In [5]:
# centroid of the casualities
casualities_centroid = centroid(casualities, features.columns.values)
print(casualities_centroid)

[2.5318761384335153, 0.8524590163934426, 30.334389799635705, 0.5537340619307832, 0.3296903460837887, 22.117886885245902, 1.6411657559198543, 0.6812386156648452]


In [0]:
# distance b/w 2 points
def point_distance(pointA, pointB):
  return np.linalg.norm(np.array(pointA)-np.array(pointB))

In [7]:
# distance b/w centroids
centroids_distance = point_distance(survivors_centroid, casualities_centroid)
print(centroids_distance)

26.365200774120844


In [0]:
# average distance between 2 points in a data set
def average_dist(data, axes):
  data = pd.DataFrame(data, columns=axes)  
  pairs = []
  for i in range(len(data)):
    for j in range(i+1, len(data)):
      pairs.append([data.iloc[i].values, data.iloc[j].values])
  
  distances = [point_distance(pointA, pointB) for (pointA, pointB) in pairs]    
  return np.array(distances).mean()

In [9]:
# average distance between each survivor
print(average_dist(survivors, features.columns.values))

61.79494312367423


In [10]:
# average distance between each casualty
print(average_dist(casualities, features.columns.values))

32.08065292422003


### Results:

*   distance b/w centroids: **26.365200774120844**
*   avg distance b/w each survivor: **61.79494312367423**
*   avg distance b/w each casualty: **32.08065292422003**   
*   what the above potentially means is that the **2 clusters are overlapping** as the distance b/w any 2 survivors is much more than the distance b/w the 2 centroids


# Next
What we want to do is to find the set of dimensions where: 

**the mean distance between each survivor and the mean distance between each casualty is less than the centroid distance.**

The set of dimensions that maximizes centroid distance and minimizes the cluster distances gives us a good shot at making clear clusters from which we could make some useful predictions.

# Part 2: Automation

### Step 1

There can be a lot of sets of dimensions, and for each of those sets, we are concerned with 3 things:

*   centroids_distance
*   avg_survivor_distance
*   avg_casualty_distance

Generate a new DataFrame (with these 3 as columns, and each unique set_of_dimension as a key), so that we can perform experiements on this dataset without having to do the distance calculations repeatedly.


In [0]:
import itertools

MAX_DIMENSIONS = len(features.columns)
MIN_DIMENSIONS = 1

def centroid_and_cluster_distances(num_of_dim):
  data = pd.DataFrame([])
  
  possible_combinations = list(itertools.combinations(np.arange(0, MAX_DIMENSIONS), num_of_dim))
  print("possible col_number combinations in " + str(num_of_dim) + " dimensions: " + str(possible_combinations))
  for each in possible_combinations:
    feature_cols = [features.columns.values[i] for i in each]
    print("========== trying feature columns: " + str(feature_cols))
    centroids_distance = point_distance(centroid(survivors, feature_cols), centroid(casualities, feature_cols))
    survivor_distance = average_dist(survivors, feature_cols)
    casualty_distance = average_dist(casualities, feature_cols)
    print("========== " + str(centroids_distance) + ", " + str(survivor_distance) + ", " + str(casualty_distance))
    data_entry = pd.DataFrame([[num_of_dim, centroids_distance, survivor_distance, casualty_distance]],
                              columns=['Dimension Size', 'Centroids Distance', 'Avg Survivor Distance', 'Avg Casualty Distance'],
                              index=[str(feature_cols)])
    data_entry.index.rename('Dimension Set', inplace=True)    
    data = data.append(data_entry)
  
  return data

In [0]:
distances = pd.DataFrame([])

for dim in range(MIN_DIMENSIONS, MAX_DIMENSIONS+1):
  distances = distances.append(centroid_and_cluster_distances(dim))

In [0]:
distances.iloc[np.r_[0:5, -5:0]]

In [0]:
from google.colab import files

def save_data_to_local(data, file):  
  with open(file, 'w') as f:
    f.write('')  
  data.to_csv(file)
  files.download(file)

In [0]:
filename = 'distances-High_Dimensional_Research.csv'

In [0]:
save_data_to_local(distances, filename)

### Load dataset from local if required (we don't want to calculate distances again and again)

In [17]:
from google.colab import files
uploaded = files.upload()
  
import io
distances = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))
distances.head()

Saving distances-High_Dimensional_Research.csv to distances-High_Dimensional_Research.csv


,Dimension Set,Dimension Size,Centroids Distance,Avg Survivor Distance,Avg Casualty Distance
0,['pclass'],1,0.581584,0.935552,0.687142
1,['sex'],1,0.533746,0.435544,0.252004
2,['age'],1,1.961320,17.230257,15.641038
3,['sibsp'],1,0.080050,0.643481,0.921636
4,['parch'],1,0.135222,0.695289,0.569875


### Step 2
Once we have our dataset of concerned points, we need to find _'the set of dimensions where: the mean distance between each survivor and the mean distance between each casualty is less than the centroid distance'_

In [18]:
optimal = distances[(distances['Avg Survivor Distance']<distances['Centroids Distance']) & (distances['Avg Casualty Distance']<distances['Centroids Distance'])]
optimal

,Dimension Set,Dimension Size,Centroids Distance,Avg Survivor Distance,Avg Casualty Distance
1,['sex'],1,0.533746,0.435544,0.252004


### Result
**['sex']** is the only set of dimensions, where this holds true: _'the mean distance between each survivor and the mean distance between each casualty is less than the centroid distance'_

## Part 2: "Best" sets of dimension
which "maximize the centroid distance and minimize the cluster distances"

### Thought process

*   ideally avg_survivor_distance/centroids_distance<1 && avg_casuality_distance/centroids_distance<1
*   what we want is to minimize both avg_survivor_distance/centroids_distance & avg_casuality_distance/centroids_distance

### Steps
*   get 2 different datasets of 20 options having smallest values of avg_survivor_distance/centroids_distance & avg_casuality_distance/centroids_distance respectively
*   and see if there are any common rows in the 2 datasets




### Implementation

In [0]:
# add 2 more columns to our dataset
distances['Survivor/Centroid ratio'] = distances['Avg Survivor Distance']/distances['Centroids Distance']
distances['Casualty/Centroid ratio'] = distances['Avg Casualty Distance']/distances['Centroids Distance']

In [20]:
distances.iloc[np.r_[0:5, -5:0]]

,Dimension Set,Dimension Size,Centroids Distance,Avg Survivor Distance,Avg Casualty Distance,Survivor/Centroid ratio,Casualty/Centroid ratio
0,['pclass'],1,0.581584,0.935552,0.687142,1.608629,1.181501
1,['sex'],1,0.533746,0.435544,0.252004,0.816014,0.472143
2,['age'],1,1.961320,17.230257,15.641038,8.785033,7.974752
3,['sibsp'],1,0.080050,0.643481,0.921636,8.038499,11.513274
4,['parch'],1,0.135222,0.695289,0.569875,5.141836,4.214367
250,"['pclass', 'sex', 'age', 'parch', 'fare', 'emb...",7,26.365079,61.776446,32.031082,2.343116,1.214906
251,"['pclass', 'sex', 'sibsp', 'parch', 'fare', 'e...",7,26.292148,54.477804,23.256386,2.072018,0.884537
252,"['pclass', 'age', 'sibsp', 'parch', 'fare', 'e...",7,26.359798,61.785197,32.071195,2.343918,1.216671
253,"['sex', 'age', 'sibsp', 'parch', 'fare', 'emba...",7,26.358785,61.775261,32.059689,2.343631,1.216281
254,"['pclass', 'sex', 'age', 'sibsp', 'parch', 'fa...",8,26.365201,61.794943,32.080653,2.343807,1.216780


In [21]:
survivor_by_centroid_min = distances.sort_values(by=['Survivor/Centroid ratio'])
survivor_by_centroid_min.head(20)

,Dimension Set,Dimension Size,Centroids Distance,Avg Survivor Distance,Avg Casualty Distance,Survivor/Centroid ratio,Casualty/Centroid ratio
1,['sex'],1,0.533746,0.435544,0.252004,0.816014,0.472143
20,"['sex', 'alone']",2,0.571627,0.805004,0.601410,1.408268,1.052102
8,"['pclass', 'sex']",2,0.789382,1.180083,0.871034,1.494946,1.103438
0,['pclass'],1,0.581584,0.935552,0.687142,1.608629,1.181501
41,"['pclass', 'sex', 'alone']",3,0.815474,1.407457,1.109347,1.725938,1.360371
16,"['sex', 'sibsp']",2,0.539715,0.943174,1.068602,1.747542,1.979937
17,"['sex', 'parch']",2,0.550608,0.996665,0.725522,1.810117,1.317675
19,"['sex', 'embarked']",2,0.603437,1.145754,0.771165,1.898715,1.277954
37,"['pclass', 'sex', 'sibsp']",3,0.793430,1.521190,1.555304,1.917231,1.960227
38,"['pclass', 'sex', 'parch']",3,0.800880,1.571051,1.252798,1.961656,1.564276


In [22]:
casualty_by_centroid_min = distances.sort_values(by=['Casualty/Centroid ratio'])
casualty_by_centroid_min.head(20)

,Dimension Set,Dimension Size,Centroids Distance,Avg Survivor Distance,Avg Casualty Distance,Survivor/Centroid ratio,Casualty/Centroid ratio
1,['sex'],1,0.533746,0.435544,0.252004,0.816014,0.472143
5,['fare'],1,26.277521,54.227378,22.897509,2.063641,0.871372
18,"['sex', 'fare']",2,26.282941,54.263853,22.935754,2.064604,0.872648
34,"['fare', 'alone']",2,26.278317,54.255923,22.934784,2.064665,0.872765
12,"['pclass', 'fare']",2,26.283956,54.268652,22.942405,2.064706,0.872867
30,"['parch', 'fare']",2,26.277869,54.283212,22.951447,2.065739,0.873414
70,"['sex', 'fare', 'alone']",3,26.283737,54.289490,22.970426,2.065516,0.873941
39,"['pclass', 'sex', 'fare']",3,26.289375,54.303010,22.979576,2.065588,0.874101
55,"['pclass', 'fare', 'alone']",3,26.284752,54.294163,22.978087,2.065614,0.874198
66,"['sex', 'parch', 'fare']",3,26.283289,54.316603,22.987297,2.066583,0.874597


In [0]:
survivor_set = survivor_by_centroid_min.iloc[0:20, 0]
casualty_set = casualty_by_centroid_min.iloc[0:20, 0]

In [24]:
# find the intersection of the 2 sets
idx1 = survivor_set.index
idx2 = casualty_set.index
common_indexes = idx1.intersection(idx2)
print(common_indexes.values)

[ 1  5 18 34 12 27 70]


In [25]:
print("'Best' sets of dimensions:")
for ind in common_indexes.values:
  print(distances.iloc[ind, 0])

'Best' sets of dimensions:
['sex']
['fare']
['sex', 'fare']
['fare', 'alone']
['pclass', 'fare']
['sibsp', 'fare']
['sex', 'fare', 'alone']
